<a href="https://www.kaggle.com/code/eavprog/absolute-fx-behavioral-profiling-k-means-cluste?scriptVersionId=304288214" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import pandas as pd
import numpy as np

# 1. Загрузка
file_path = '/kaggle/input/notebooks/eavprog/abscur2/abscur.csv'
df = pd.read_csv(file_path, parse_dates=['Date'])
df = df.sort_values('Date').set_index('Date')

# 2. Очистка (ML-Ready)
# Удаляем валюты, где вообще нет данных (если такие есть)
df = df.dropna(axis=1, how='all')

# Заполняем пропуски: сначала вперед (последним известным), потом назад (для самого начала истории)
df = df.ffill().bfill()

# Проверка на критические ошибки
if df.isnull().values.any():
    print("⚠️ Внимание: в данных остались пропуски!")
else:
    print("✅ Данные очищены и готовы к анализу.")

print(f"Итоговая матрица: {df.shape[0]} дней x {df.shape[1]} валют")

✅ Данные очищены и готовы к анализу.
Итоговая матрица: 5898 дней x 45 валют


## 1.1. Анализ изменчивости логарифмической доходности (Volatility)

На первом этапе подготовки данных мы рассчитываем **годовую волатильность логарифмических доходностей**. 

### Почему это важно для машинного обучения?
В отличие от анализа номинальных курсов, работа с доходностями позволяет алгоритму **K-Means** сравнивать активы разного порядка стоимости. Логарифмирование нормализует распределение, сглаживая экстремальные выбросы, что критично для корректной работы метрических алгоритмов кластеризации.

**Технические детали расчета:**
* Мы используем стандартное отклонение ежедневных логарифмических доходностей.
* Результат масштабируется к годовому значению (умножение на $\sqrt{252}$), что позволяет интерпретировать риск в привычных для финансового анализа величинах.
* **Важное отличие:** На сайте проекта представлена страница [Рейтинги волатильности](https://www.abscur.ru/p/blog-page_26.html), где используется расчет на основе *арифметической* доходности без приведения к годовому значению. Здесь же мы намеренно переходим к годовой логарифмической волатильности, чтобы подготовить данные для обучения модели.

В таблицах ниже представлены крайние значения выборки. Тикеры являются кликабельными и ведут на интерактивные графики соответствующих абсолютных курсов на сайте проекта **Abscur**.

In [2]:
import numpy as np
import pandas as pd
from IPython.display import HTML

# Создаем базовый DataFrame для признаков (индекс — тикеры валют)
features_df = pd.DataFrame(index=df.columns)

# 1. Вычисляем ежедневные логарифмические доходности
# log(P_t / P_{t-1})
log_returns = np.log(df / df.shift(1))

# 2. Считаем стандартное отклонение ежедневных доходностей и масштабируем к году (sqrt(252))
features_df['Volatility'] = log_returns.std() * np.sqrt(252)

# Функция для создания кликабельных ссылок на абсолютные курсы
def make_clickable(ticker):
    url = f"https://www.abscur.ru/p/2.html?abs={ticker}"
    return f'<a href="{url}" target="_blank">{ticker}</a>'

# Подготовка таблицы для вывода (копируем, чтобы не портить основной df ссылками)
display_df = features_df[['Volatility']].copy()
display_df['Volatility'] = (display_df['Volatility'] * 100).round(2).astype(str) + '%'
display_df.index = [make_clickable(t) for t in display_df.index]

print("Топ-5 валют с минимальной изменчивостью доходности:")
display(HTML(display_df.sort_values('Volatility').head(5).to_html(escape=False)))

print("\nТоп-5 валют с максимальной изменчивостью доходности:")
display(HTML(display_df.sort_values('Volatility', ascending=False).head(5).to_html(escape=False)))

Топ-5 валют с минимальной изменчивостью доходности:


,Volatility
BRL,10.13%
TRY,11.38%
RUB,12.09%
EGP,15.45%
UAH,16.08%



Топ-5 валют с максимальной изменчивостью доходности:


,Volatility
ZAR,8.89%
KZT,8.89%
COP,8.82%
CLP,7.73%
MXN,7.52%
